# Missing Values and Split Apply Combine

This lesson focuses on reviewing our basics with pandas and extending them to more advanced munging and cleaning.  Specifically, we will discuss how to load data files, work with missing values, use split-apply-combine, use string methods, and work with string and datetime objects.  By the end of this lesson you should feel confident doing basic exploratory data analysis using `pandas`. 

**OBJECTIVES**

- Read `.csv` files in as `DataFrame` objects
- Drop missing values
- Replace missing values
- Impute missing values
- Use `.groupby` and split-apply-combine
- Use `.transform` together with `.groupby`
- Use `.unstack` to unstack grouped data


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Missing Values

Missing values are a common problem in data, whether this is because they are truly missing or there is confusion between the data encoding and the methods you read the data in using.

In [4]:
ufo_url = 'https://raw.githubusercontent.com/jfkoehler/nyu_bootcamp_fa25/refs/heads/main/data/ufo.csv'

In [5]:
#create ufo dataframe


In [6]:
# examine ufo info


In [7]:
# one-liner to count missing values


In [8]:
# drop missing values


In [9]:
# still there?


In [10]:
# fill missing values


In [11]:
# most common values as a dictionary


In [12]:
# replace missing values with most common value


#### Problem

1. Read in the dataset `churn_missing.csv` from our repo as a `DataFrame` using the url below, assign to a variable `churn_df`

In [13]:
churn_url = 'https://raw.githubusercontent.com/jfkoehler/nyu_bootcamp_fa25/refs/heads/main/data/churn_missing.csv'
churn_df = ''

2. Are there any missing values?  What columns are they in and how many are there?

3. What do you think we should do about these?  Drop, replace, impute?

### `groupby`

Often, you are faced with a dataset that you are interested in summaries within groups based on a condition.  The simplest condition is that of a unique value in a single column.  Using `.groupby` you can split your data into unique groups and summarize the results.  

**NOTE**: After splitting you need to summarize!

![](https://www.oreilly.com/api/v2/epubs/9781783985128/files/graphics/5128OS_09_01.jpg)

In [14]:
# sample data
titanic = sns.load_dataset('titanic')
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [40]:
# survival rate of each sex?


In [41]:
# survival rate of each class?


In [42]:
# each class and sex survival rate


In [43]:
# working with multi-index -- changing form of results


In [44]:
# age less than 40 survival rate


In [ ]:
less_than_40 = ''


#### A Little More with `groupby`

So far, each example applies one summary to each group. We can also ask for more than one summary at a time with `.agg()`.

For example, the survival rate is more useful when we can see how many passengers are in each class. Here, `count` gives the number of observations and `mean` gives the survival rate.


In [15]:
titanic.groupby('class')['survived'].agg(['count', 'mean'])

/var/folders/8v/7bhy8yqn04b7rzqglb2s38200000gn/T/ipykernel_14976/4132044711.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  titanic.groupby('class')['survived'].agg(['count', 'mean'])


,count,mean
class,,
First,216,0.629630
Second,184,0.472826
Third,491,0.242363


In [16]:
class_summary = titanic.groupby('class')['survived'].agg(['count', 'mean'])
class_summary


/var/folders/8v/7bhy8yqn04b7rzqglb2s38200000gn/T/ipykernel_14976/1018959238.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  class_summary = titanic.groupby('class')['survived'].agg(['count', 'mean'])


,count,mean
class,,
First,216,0.629630
Second,184,0.472826
Third,491,0.242363


A regular `groupby` summary produces one value for each group. Sometimes we want to keep every original row and place the group average beside it. `.transform('mean')` does this without changing the number of rows.


In [26]:
titanic.groupby('class', observed = False).agg(
    survival_rate = ('survived', 'mean'),
    avg_fare = ('fare', 'mean'),
    median_age = ('age', 'median'))

,survival_rate,avg_fare,median_age
class,,,
First,0.629630,84.154687,37.0
Second,0.472826,20.662183,29.0
Third,0.242363,13.675550,24.0


In [27]:
titanic.groupby('class', observed = False)['survived'].transform('mean')

0      0.242363
1      0.629630
2      0.242363
3      0.629630
4      0.242363
         ...   
886    0.472826
887    0.629630
888    0.242363
889    0.629630
890    0.242363
Name: survived, Length: 891, dtype: float64

In [28]:
titanic['class_survival_rate'] = titanic.groupby('class', observed = False)['survived'].transform('mean')

titanic[['class', 'survived', 'class_survival_rate']].head(10)


,class,survived,class_survival_rate
0,Third,0,0.242363
1,First,1,0.629630
2,Third,1,0.242363
3,First,1,0.629630
4,Third,0,0.242363
5,Third,0,0.242363
6,First,0,0.629630
7,Third,0,0.242363
8,Third,1,0.242363
9,Second,1,0.472826


In [22]:
titanic.groupby(['class', 'sex'])['survived'].mean()

/var/folders/8v/7bhy8yqn04b7rzqglb2s38200000gn/T/ipykernel_14976/1827838915.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  titanic.groupby(['class', 'sex'])['survived'].mean()


class   sex   
First   female    0.968085
        male      0.368852
Second  female    0.921053
        male      0.157407
Third   female    0.500000
        male      0.135447
Name: survived, dtype: float64

In [24]:
titanic.groupby(['class', 'sex'])['survived'].mean().unstack('sex')

/var/folders/8v/7bhy8yqn04b7rzqglb2s38200000gn/T/ipykernel_14976/4186619017.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  titanic.groupby(['class', 'sex'])['survived'].mean().unstack('sex')


sex,female,male
class,,
First,0.968085,0.368852
Second,0.921053,0.157407
Third,0.500000,0.135447


**Quick check:**

- What does one row of `class_summary` represent?
- Why does `class_survival_rate` have the same number of values as the original Titanic dataset?
- When would you choose a group summary, and when would you choose `transform`?


#### In-Class Activity: Restaurant Tips

Work with a partner for approximately 20 minutes. Use the `tips` dataset to practice asking a question, choosing a grouping column, and selecting an appropriate summary.

For each result, be prepared to explain:

- what one row of the result represents;
- which column or columns define the groups;
- which calculation was applied after grouping;
- one conclusion that the restaurant manager might investigate further.


In [ ]:
tips = sns.load_dataset('tips')

In [ ]:
tips.head(2)

1. Find the average tip for smokers and non-smokers. Which group has the larger average tip? Write one sentence interpreting the result.


2. Find the average total bill for each combination of day and time. What does one row of this result represent?


3. Write one additional question about the restaurant that `groupby` could help answer. Then write the code needed to answer it.


4. Display both the number of checks and the average tip for each day. Why is the count useful when interpreting the mean?


In [ ]:
# your code here


5. Add a column named `day_average_bill` using:

```python
tips.groupby('day')['total_bill'].transform('mean')
```

Then add a second column showing the difference between each check's `total_bill` and its day average. Display the four columns `day`, `total_bill`, `day_average_bill`, and your difference column.


In [ ]:
# your code here


**Activity wrap-up**

Choose one of your grouped results and write a two- or three-sentence note to the restaurant manager:

1. describe one pattern;
2. support it with a number from your result;
3. name one limitation or additional question.


### Upcoming Quiz and Project

- **Small group project**: Next week, you will begin work in small groups to explore a real-world dataset and develop a short, evidence-based recommendation for a stakeholder. Each group will identify a focused question, inspect and clean the data—including documenting any decisions about missing values—and use pandas tools such as filtering, groupby, aggregation, and basic visualization to investigate it. Your group will share a brief summary of its question, methods, key findings, one limitation, and a recommended next step, supported by clearly labeled tables or plots.

- **Pandas Quiz**: Short in class quiz to assess your understanding of pandas.  Topics to include:

     - Selecting column(s)
     - Filtering rows with `.loc`
     - Grouping and summarizing data with `.groupby`

### Exit Ticket

Please complete the exit ticket for today [here](https://forms.gle/JGrskr1QVYYksZ959).

### Data Resources

- NYU has a number of resources for acquiring data with applications to economics and finance [here](https://guides.nyu.edu/finance).
